In [1]:
# Sea Ice Dataset — K-Means + NLP

**CSV file:** `seaice.csv`
**Jupyter Notebook:** `seaice_kmeans_nlp_analysis.ipynb`

```python
# ============================================================
# SEA ICE DATASET ANALYSIS
# K-MEANS CLUSTERING + NLP
# ============================================================

# ============================================================
# CELL 1 — IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import warnings

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")
```

```python
# ============================================================
# CELL 2 — LOAD DATASET
# ============================================================

# Make sure seaice.csv is in the same folder as this notebook

file_name = "seaice.csv"

df = pd.read_csv(file_name)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

display(df.head())
```

```python
# ============================================================
# CELL 3 — EXPLORE THE DATASET
# ============================================================

print("Column names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nStatistical summary:")
display(df.describe(include="all"))
```

```python
# ============================================================
# CELL 4 — CHECK MISSING VALUES
# ============================================================

print("Missing values in each column:")

missing_values = df.isnull().sum()

display(missing_values)

print("\nTotal missing values:", df.isnull().sum().sum())
```

```python
# ============================================================
# CELL 5 — REMOVE DUPLICATES
# ============================================================

before = len(df)

df = df.drop_duplicates()

after = len(df)

print("Duplicate rows removed:", before - after)
print("New dataset shape:", df.shape)
```

```python
# ============================================================
# CELL 6 — IDENTIFY NUMERIC AND TEXT COLUMNS
# ============================================================

numeric_columns = df.select_dtypes(
    include=np.number
).columns.tolist()

text_columns = df.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numeric columns:")
print(numeric_columns)

print("\nText columns:")
print(text_columns)
```

```python
# ============================================================
# CELL 7 — PREPARE NUMERIC DATA FOR K-MEANS
# ============================================================

if len(numeric_columns) == 0:
    print("No numeric columns were found.")
else:

    numeric_data = df[numeric_columns].copy()

    # Replace missing numeric values with median
    for column in numeric_columns:
        numeric_data[column] = numeric_data[column].fillna(
            numeric_data[column].median()
        )

    print("Numeric data after missing-value treatment:")
    display(numeric_data.head())

    # Standardization
    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(numeric_data)

    print("Standardization completed.")
    print("Shape of scaled data:", X_scaled.shape)
```

```python
# ============================================================
# CELL 8 — ELBOW METHOD
# ============================================================

if len(numeric_columns) > 0:

    # Maximum K should not exceed number of observations
    max_k = min(10, len(df) - 1)

    if max_k >= 2:

        k_values = range(2, max_k + 1)

        inertia_values = []

        for k in k_values:

            model = KMeans(
                n_clusters=k,
                random_state=42,
                n_init=10
            )

            model.fit(X_scaled)

            inertia_values.append(model.inertia_)

        plt.figure(figsize=(8, 5))

        plt.plot(
            list(k_values),
            inertia_values,
            marker="o"
        )

        plt.xlabel("Number of Clusters (K)")
        plt.ylabel("Inertia")
        plt.title("Elbow Method")

        plt.grid(True)
        plt.show()

    else:
        print("Not enough observations for K-Means.")
```

```python
# ============================================================
# CELL 9 — SILHOUETTE SCORE
# ============================================================

if len(numeric_columns) > 0:

    max_k = min(10, len(df) - 1)

    if max_k >= 2:

        k_values = range(2, max_k + 1)

        silhouette_values = []

        for k in k_values:

            model = KMeans(
                n_clusters=k,
                random_state=42,
                n_init=10
            )

            labels = model.fit_predict(X_scaled)

            score = silhouette_score(
                X_scaled,
                labels
            )

            silhouette_values.append(score)

        plt.figure(figsize=(8, 5))

        plt.plot(
            list(k_values),
            silhouette_values,
            marker="o"
        )

        plt.xlabel("Number of Clusters (K)")
        plt.ylabel("Silhouette Score")
        plt.title("Silhouette Score by Number of Clusters")

        plt.grid(True)
        plt.show()

        best_k = list(k_values)[
            np.argmax(silhouette_values)
        ]

        print("Recommended K:", best_k)

    else:
        best_k = 2
```

```python
# ============================================================
# CELL 10 — APPLY K-MEANS
# ============================================================

if len(numeric_columns) > 0:

    kmeans = KMeans(
        n_clusters=best_k,
        random_state=42,
        n_init=10
    )

    cluster_labels = kmeans.fit_predict(X_scaled)

    df["KMeans_Cluster"] = cluster_labels

    print("K-Means clustering completed.")

    display(df.head())
```

```python
# ============================================================
# CELL 11 — CLUSTER COUNTS
# ============================================================

if "KMeans_Cluster" in df.columns:

    cluster_counts = (
        df["KMeans_Cluster"]
        .value_counts()
        .sort_index()
    )

    print("Number of records in each cluster:")

    display(cluster_counts)
```

```python
# ============================================================
# CELL 12 — CLUSTER SUMMARY
# ============================================================

if "KMeans_Cluster" in df.columns:

    cluster_summary = (
        df.groupby("KMeans_Cluster")[numeric_columns]
        .mean()
    )

    print("Average values for each cluster:")

    display(cluster_summary)
```

```python
# ============================================================
# CELL 13 — PCA VISUALIZATION OF K-MEANS
# ============================================================

if len(numeric_columns) > 0:

    pca = PCA(n_components=2)

    X_pca = pca.fit_transform(X_scaled)

    plt.figure(figsize=(9, 6))

    scatter = plt.scatter(
        X_pca[:, 0],
        X_pca[:, 1],
        c=df["KMeans_Cluster"],
        alpha=0.7
    )

    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")

    plt.title(
        "Sea Ice Dataset — K-Means Clusters"
    )

    plt.colorbar(
        scatter,
        label="Cluster"
    )

    plt.grid(True)
    plt.show()

    print(
        "Explained variance:",
        pca.explained_variance_ratio_
    )
```

```python
# ============================================================
# CELL 14 — CREATE TEXT DATA FOR NLP
# ============================================================

if len(text_columns) > 0:

    # Combine all text columns into one column

    df["combined_text"] = (
        df[text_columns]
        .fillna("")
        .astype(str)
        .agg(" ".join, axis=1)
    )

    print("Text columns combined successfully.")

    display(
        df[["combined_text"]].head()
    )

else:

    print(
        "No text columns were found in the dataset."
    )
```

```python
# ============================================================
# CELL 15 — TEXT CLEANING
# ============================================================

def clean_text(text):

    text = str(text).lower()

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+",
        "",
        text
    )

    # Remove numbers
    text = re.sub(
        r"\d+",
        " ",
        text
    )

    # Remove punctuation
    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


if "combined_text" in df.columns:

    df["clean_text"] = (
        df["combined_text"]
        .apply(clean_text)
    )

    print("Text cleaning completed.")

    display(
        df[
            ["combined_text", "clean_text"]
        ].head()
    )
```

```python
# ============================================================
# CELL 16 — TF-IDF
# ============================================================

if "clean_text" in df.columns:

    # Remove empty text rows from NLP processing
    text_for_nlp = df["clean_text"].fillna("")

    tfidf = TfidfVectorizer(
        stop_words="english",
        max_features=1000,
        min_df=1
    )

    X_text = tfidf.fit_transform(
        text_for_nlp
    )

    print("TF-IDF completed.")

    print(
        "TF-IDF matrix shape:",
        X_text.shape
    )
```

```python
# ============================================================
# CELL 17 — DISPLAY TF-IDF WORDS
# ============================================================

if "clean_text" in df.columns:

    words = tfidf.get_feature_names_out()

    print("Number of words/features:", len(words))

    print("\nFirst 50 features:")

    print(words[:50])
```

```python
# ============================================================
# CELL 18 — NLP K-MEANS
# ============================================================

if "clean_text" in df.columns:

    number_of_documents = X_text.shape[0]

    if number_of_documents >= 2:

        # Use the same K as numerical clustering where possible
        nlp_k = min(
            best_k if "best_k" in globals() else 3,
            number_of_documents
        )

        if nlp_k < 2:
            nlp_k = 2

        nlp_kmeans = KMeans(
            n_clusters=nlp_k,
            random_state=42,
            n_init=10
        )

        nlp_labels = nlp_kmeans.fit_predict(
            X_text
        )

        df["NLP_Cluster"] = nlp_labels

        print(
            "NLP K-Means completed."
        )

        print(
            "Number of NLP clusters:",
            nlp_k
        )

        display(
            df[
                ["clean_text", "NLP_Cluster"]
            ].head(10)
        )
```

```python
# ============================================================
# CELL 19 — NLP CLUSTER COUNTS
# ============================================================

if "NLP_Cluster" in df.columns:

    nlp_counts = (
        df["NLP_Cluster"]
        .value_counts()
        .sort_index()
    )

    print(
        "Number of documents in each NLP cluster:"
    )

    display(nlp_counts)
```

```python
# ============================================================
# CELL 20 — MOST IMPORTANT WORDS IN EACH NLP CLUSTER
# ============================================================

if "NLP_Cluster" in df.columns:

    feature_names = np.array(
        tfidf.get_feature_names_out()
    )

    print(
        "Important words in each NLP cluster:"
    )

    for cluster_number in range(nlp_k):

        center = (
            nlp_kmeans
            .cluster_centers_[cluster_number]
        )

        top_indices = center.argsort()[-15:][::-1]

        top_words = feature_names[
            top_indices
        ]

        print(
            f"\nNLP Cluster {cluster_number}"
        )

        print(
            ", ".join(top_words)
        )
```

```python
# ============================================================
# CELL 21 — NLP SILHOUETTE SCORE
# ============================================================

if "NLP_Cluster" in df.columns:

    # Silhouette score requires at least 2 clusters
    if len(set(nlp_labels)) > 1:

        nlp_silhouette = silhouette_score(
            X_text,
            nlp_labels,
            metric="cosine"
        )

        print(
            "NLP Silhouette Score:",
            round(nlp_silhouette, 4)
        )
```

```python
# ============================================================
# CELL 22 — VISUALIZE NLP CLUSTERS
# ============================================================

if "NLP_Cluster" in df.columns:

    # PCA requires a dense matrix
    X_text_dense = X_text.toarray()

    pca_text = PCA(
        n_components=2,
        random_state=42
    )

    X_text_pca = pca_text.fit_transform(
        X_text_dense
    )

    plt.figure(figsize=(9, 6))

    scatter = plt.scatter(
        X_text_pca[:, 0],
        X_text_pca[:, 1],
        c=df["NLP_Cluster"],
        alpha=0.7
    )

    plt.xlabel(
        "Text Principal Component 1"
    )

    plt.ylabel(
        "Text Principal Component 2"
    )

    plt.title(
        "Sea Ice Dataset — NLP K-Means Clusters"
    )

    plt.colorbar(
        scatter,
        label="NLP Cluster"
    )

    plt.grid(True)

    plt.show()
```

```python
# ============================================================
# CELL 23 — COMPARE NUMERICAL AND NLP CLUSTERS
# ============================================================

if (
    "KMeans_Cluster" in df.columns
    and "NLP_Cluster" in df.columns
):

    comparison = pd.crosstab(
        df["KMeans_Cluster"],
        df["NLP_Cluster"]
    )

    print(
        "Comparison of K-Means and NLP clusters:"
    )

    display(comparison)
```

```python
# ============================================================
# CELL 24 — DISPLAY RECORDS BY CLUSTER
# ============================================================

if "KMeans_Cluster" in df.columns:

    for cluster in sorted(
        df["KMeans_Cluster"].unique()
    ):

        print(
            f"\n===== K-MEANS CLUSTER {cluster} ====="
        )

        cluster_data = df[
            df["KMeans_Cluster"] == cluster
        ]

        display(
            cluster_data.head(10)
        )
```

```python
# ============================================================
# CELL 25 — FINAL DATASET
# ============================================================

print("Final dataset shape:", df.shape)

print("\nFinal columns:")

print(df.columns.tolist())

display(df.head(10))
```

```python
# ============================================================
# CELL 26 — SAVE RESULTS
# ============================================================

output_file = "seaice_kmeans_nlp_results.csv"

df.to_csv(
    output_file,
    index=False
)

print(
    "Analysis completed successfully."
)

print(
    "Results saved as:",
    output_file
)
```

```python
# ============================================================
# CELL 27 — FINAL SUMMARY
# ============================================================

print("=" * 60)
print("SEA ICE K-MEANS + NLP ANALYSIS SUMMARY")
print("=" * 60)

print(
    "\nOriginal dataset shape:",
    df.shape
)

if "KMeans_Cluster" in df.columns:

    print(
        "\nNumber of K-Means clusters:",
        df["KMeans_Cluster"].nunique()
    )

    print(
        "K-Means cluster sizes:"
    )

    print(
        df["KMeans_Cluster"]
        .value_counts()
        .sort_index()
    )

if "NLP_Cluster" in df.columns:

    print(
        "\nNumber of NLP clusters:",
        df["NLP_Cluster"].nunique()
    )

    print(
        "NLP cluster sizes:"
    )

    print(
        df["NLP_Cluster"]
        .value_counts()
        .sort_index()
    )

print("\nOutput file:")
print("seaice_kmeans_nlp_results.csv")

print("\nAnalysis finished.")
```

## Required files

Place both files in the same folder:

```text
your_folder/
│
├── seaice.csv
└── seaice_kmeans_nlp_analysis.ipynb
```

After running the notebook, it will create:

```text
seaice_kmeans_nlp_results.csv
```

**Note:** The NLP section only becomes active if `seaice.csv` contains text columns. If your sea-ice dataset contains only numerical variables such as year, month, extent, area, concentration, latitude, or longitude, the K-Means section will run normally but there may be no meaningful NLP analysis.


SyntaxError: invalid character '│' (U+2502) (1900576484.py, line 793)